# Home Credit Default Risk
## Sprint 1 — Definição do Problema e Análise Exploratória de Dados

**Dataset:** `application_train.csv` (~307k linhas, 122 colunas)  
**Target:** `TARGET` — 0 = pagou normalmente, 1 = dificuldade de pagamento nas parcelas iniciais  
**Tipo de tarefa:** Classificação binária  

---

## Seção 1 — Apresentação do Projeto

### Pergunta de negócio

No momento da solicitação de crédito, quais clientes têm risco de inadimplência alto o suficiente para justificar negação automática ou revisão manual, considerando que rejeitar um bom pagador tem custo de oportunidade e aprovar um mau pagador tem custo de perda financeira direta?

### Por que accuracy não é a métrica certa

A base tem ~8% de inadimplentes. Um modelo trivial que prevê 0 para todos os casos entrega 92% de accuracy sem nenhum valor preditivo. O objetivo é construir um modelo cujas métricas reflitam o tradeoff assimétrico entre falso negativo (aprovar inadimplente = perda financeira direta) e falso positivo (negar bom pagador = custo de oportunidade).

### Escopo desta sprint

Entender a estrutura dos dados disponíveis no momento da decisão de crédito — sem tratamento de missings, encoding, feature engineering ou treinamento de modelo. Isso fica para sprints posteriores.

## Seção 2 — Carregamento e Visão Geral

Tópicos a cobrir:

- Carregamento de `data/raw/application_train.csv`
- Shape do dataframe
- Tipos de colunas (numéricas vs categóricas)
- Percentual de valores nulos por feature — heatmap e ranking
- Primeiras linhas e estatísticas descritivas básicas (`describe()`)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

# 1. CARREGAMENTO DO DATASET
df = pd.read_csv('../data/raw/application_train.csv')

print("=" * 80)
print("1. CARREGAMENTO E SHAPE")
print("=" * 80)
print(f"\nDataset: application_train.csv")
print(f"Shape: {df.shape[0]:,} linhas × {df.shape[1]} colunas")

# 2. TIPOS DE COLUNAS
print("\n" + "=" * 80)
print("2. TIPOS DE COLUNAS")
print("=" * 80)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"\nNuméricas: {len(numeric_cols)}")
print(f"Categóricas: {len(categorical_cols)}")
print(f"\nDetalhes dos tipos:")
print(df.dtypes.value_counts())

# 3. VALORES NULOS - RANKING
print("\n" + "=" * 80)
print("3. PERCENTUAL DE VALORES NULOS POR FEATURE")
print("=" * 80)

missing_data = pd.DataFrame({
    'Feature': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percent': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('Missing_Percent', ascending=False).reset_index(drop=True)

print(f"\nTotal de features com missing: {(missing_data['Missing_Count'] > 0).sum()}")
print("\nRanking das features com maior % de nulos:")
print(missing_data[missing_data['Missing_Count'] > 0].to_string(index=False))

# 4. HEATMAP DE VALORES NULOS
print("\n" + "=" * 80)
print("VISUALIZAÇÃO: Heatmap de Valores Nulos")
print("=" * 80)

fig, ax = plt.subplots(figsize=(18, 8))
missing_mask = df.isnull().T
missing_mask = missing_mask[missing_mask.sum(axis=1) > 0]
missing_mask = missing_mask.loc[missing_mask.sum(axis=1).sort_values(ascending=False).index]
sns.heatmap(missing_mask, cbar=True, yticklabels=True, xticklabels=False, ax=ax, cmap='RdYlGn_r')
ax.set_title('Padrão de Valores Nulos (Features com Missing > 0)', fontsize=14, fontweight='bold')
ax.set_ylabel('Features', fontsize=11)
plt.tight_layout()
plt.show()

# 5. PRIMEIRAS LINHAS
print("\n" + "=" * 80)
print("4. PRIMEIRAS 5 LINHAS")
print("=" * 80)
print(df.head())

# 6. ESTATÍSTICAS DESCRITIVAS
print("\n" + "=" * 80)
print("5. ESTATÍSTICAS DESCRITIVAS BÁSICAS")
print("=" * 80)
print(df.describe().T)

## Seção 3 — Análise Univariada

Tópicos a cobrir:

- Distribuição do target (`TARGET`) — proporção 0/1, implicações para modelagem
- Distribuição das principais features numéricas (histogramas, detecção de outliers)
- Distribuição das features categóricas (frequência de categorias)
- Identificação de colunas com baixa variância ou alta concentração em uma categoria

In [ ]:
print("=" * 80)
print("1. DISTRIBUIÇÃO DO TARGET")
print("=" * 80)

target_counts = df['TARGET'].value_counts().sort_index()
target_pct = (target_counts / len(df) * 100).round(2)

target_summary = pd.DataFrame({
    'Classe': ['0 — Pagou normalmente', '1 — Dificuldade de pagamento'],
    'Contagem': target_counts.values,
    'Percentual (%)': target_pct.values
})
print(f"\nTotal de registros: {len(df):,}")
print(target_summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    ['0 — Pagou normalmente', '1 — Dificuldade de pagamento'],
    target_counts.values,
    color=['#2ecc71', '#e74c3c'],
    edgecolor='white',
    linewidth=1.2
)
for bar, pct in zip(bars, target_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2000,
            f'{pct}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_title('Distribuição do Target — Desbalanceamento de Classes', fontsize=14, fontweight='bold')
ax.set_ylabel('Quantidade de Registros', fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("2. DISTRIBUIÇÃO DAS PRINCIPAIS FEATURES NUMÉRICAS")
print("=" * 80)

top_numeric = [
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY',
    'AMT_GOODS_PRICE', 'DAYS_BIRTH', 'DAYS_EMPLOYED',
    'DAYS_REGISTRATION', 'CNT_CHILDREN', 'REGION_POPULATION_RELATIVE'
]
top_numeric = [c for c in top_numeric if c in df.columns]

print(f"\nFeatures selecionadas: {top_numeric}")
print("\nEstatísticas descritivas:")
print(df[top_numeric].describe().T[['mean', 'std', 'min', '50%', 'max']].round(2).to_string())

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(top_numeric):
    data = df[col].dropna()
    axes[i].hist(data, bins=50, color='#3498db', edgecolor='white', linewidth=0.5)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Frequência', fontsize=9)
    axes[i].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    axes[i].tick_params(axis='x', rotation=30, labelsize=8)

for j in range(len(top_numeric), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribuição das Principais Features Numéricas', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("3. DISTRIBUIÇÃO DAS FEATURES CATEGÓRICAS")
print("=" * 80)

categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"\nTotal de features categóricas: {len(categorical_cols)}")

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(categorical_cols[:9]):
    freq = df[col].value_counts()
    axes[i].barh(freq.index[:10], freq.values[:10], color='#9b59b6', edgecolor='white', linewidth=0.5)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Frequência', fontsize=9)
    axes[i].tick_params(axis='y', labelsize=8)
    axes[i].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))

for j in range(min(len(categorical_cols), 9), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribuição das Features Categóricas (Top 10 Categorias por Feature)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("4. FEATURES COM ALTA CONCENTRAÇÃO EM UMA CATEGORIA")
print("=" * 80)

concentration = []
for col in df.columns:
    top_val_pct = df[col].value_counts(normalize=True).iloc[0] * 100
    concentration.append({'Feature': col, 'Valor_Dominante': df[col].value_counts().index[0], 'Concentracao_%': round(top_val_pct, 2)})

concentration_df = pd.DataFrame(concentration).sort_values('Concentracao_%', ascending=False).reset_index(drop=True)
high_concentration = concentration_df[concentration_df['Concentracao_%'] >= 90]

print(f"\nFeatures com ≥ 90% dos registros em um único valor: {len(high_concentration)}")
print(high_concentration.to_string(index=False))


## Seção 4 — Análise Bivariada e Multivariada

Tópicos a cobrir:

- Correlação entre features numéricas (heatmap)
- Relação entre features e TARGET — boxplots, violin plots, proporção de inadimplência por categoria
- Features com maior poder discriminativo aparente
- Multicolinearidade — pares de features altamente correlacionadas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

# assume que df foi carregado na Seção 2 — se precisar recarregar:
# df = pd.read_csv('../data/raw/application_train.csv')

# garante que TARGET é int independente do que a Seção 2 fez
df['TARGET'] = df['TARGET'].astype(int)

num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

print(f'Features numéricas: {len(num_cols)}')
print(f'Features categóricas: {len(cat_cols)}')

In [ ]:
# 4.1 Correlação com TARGET — ranking das features numéricas ---

corr_target = (
    df[num_cols]
    .corr()['TARGET']
    .drop('TARGET')
    .sort_values(key=abs, ascending=False)
)

TOP_N = 20
top_features = corr_target.head(TOP_N)

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#d73027' if v > 0 else '#4575b4' for v in top_features.values]
ax.barh(top_features.index[::-1], top_features.values[::-1], color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlação de Pearson com TARGET')
ax.set_title(f'Top {TOP_N} features por correlação absoluta com TARGET')
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.show()

print('\nTop 10 correlações positivas (maior risco):')
print(corr_target[corr_target > 0].head(10).round(4).to_string())
print('\nTop 10 correlações negativas (menor risco):')
print(corr_target[corr_target < 0].head(10).round(4).to_string())

In [ ]:
# 4.2 Heatmap de correlação entre as top features ---

top_feat_names = corr_target.head(15).index.tolist()
cols_heatmap = top_feat_names + ['TARGET']

corr_matrix = df[cols_heatmap].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax,
)
ax.set_title('Heatmap de correlação — top 15 features + TARGET', pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# 4.3 Boxplots: distribuição das top features numéricas por TARGET ---

top5 = corr_target.head(5).index.tolist()

fig, axes = plt.subplots(1, len(top5), figsize=(4 * len(top5), 5))

for ax, feat in zip(axes, top5):
    data = df[[feat, 'TARGET']].dropna().copy()
    data['TARGET'] = data['TARGET'].astype(int)
    p1, p99 = data[feat].quantile([0.01, 0.99])
    data = data[data[feat].between(p1, p99)]

    sns.boxplot(
        data=data, x='TARGET', y=feat,
        order=[0, 1],
        palette=['#4575b4', '#d73027'],  # lista posicional: evita bug de chaves no seaborn < 0.13
        width=0.5, ax=ax,
    )
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel('')
    ax.set_xticklabels(['Adimplente', 'Inadimplente'], fontsize=8)

plt.suptitle('Distribuição das top 5 features por TARGET (sem outliers extremos)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 4.4 Taxa de inadimplência por feature categórica ---

cat_plot = [c for c in cat_cols if 2 <= df[c].nunique() <= 20]

cat_stats = {}
for col in cat_plot:
    rates = df.groupby(col)['TARGET'].mean().sort_values(ascending=False)
    cat_stats[col] = {
        'rates': rates,
        'spread': rates.max() - rates.min(),
    }

cat_sorted = sorted(cat_stats.items(), key=lambda x: x[1]['spread'], reverse=True)

top_cats = cat_sorted[:4]
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for ax, (col, stats) in zip(axes, top_cats):
    rates = stats['rates']
    counts = df[col].value_counts()
    label = [f"{cat}\n(n={counts.get(cat, 0):,})" for cat in rates.index]

    ax.bar(range(len(rates)), rates.values, color='#4575b4', alpha=0.85)
    ax.axhline(df['TARGET'].mean(), color='red', linestyle='--', linewidth=1,
               label=f'Média geral ({df["TARGET"].mean():.1%})')
    ax.set_xticks(range(len(rates)))
    ax.set_xticklabels(label, fontsize=8, rotation=20, ha='right')
    ax.set_ylabel('Taxa de inadimplência')
    ax.set_title(f'{col}  (spread={stats["spread"]:.3f})', fontsize=10)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
    ax.legend(fontsize=8)

plt.suptitle('Taxa de inadimplência por categoria — top 4 features mais discriminativas', y=1.01)
plt.tight_layout()
plt.show()

print('\nRanking completo — spread de inadimplência por feature categórica:')
spread_df = pd.DataFrame([
    {'Feature': col, 'Spread (max-min)': round(s['spread'], 4),
     'Taxa máx': round(s['rates'].max(), 4), 'Taxa mín': round(s['rates'].min(), 4)}
    for col, s in cat_sorted
])
display(spread_df)

In [ ]:
# 4.5 Multicolinearidade — pares com |corr| > 0.7 ---

THRESHOLD = 0.7

corr_full = df[num_cols].drop(columns=['TARGET'], errors='ignore').corr().abs()
upper = corr_full.where(np.triu(np.ones(corr_full.shape), k=1).astype(bool))

high_corr = (
    upper.stack()
    .reset_index()
    .rename(columns={'level_0': 'Feature A', 'level_1': 'Feature B', 0: '|Correlação|'})
    .query('`|Correlação|` >= @THRESHOLD')
    .sort_values('|Correlação|', ascending=False)
    .reset_index(drop=True)
)

print(f'Pares com |correlação| >= {THRESHOLD}: {len(high_corr)}')
display(high_corr.head(20).round(4))

if len(high_corr) > 0:
    mc_features = pd.unique(high_corr[['Feature A', 'Feature B']].values.ravel()).tolist()[:20]
    corr_mc = df[mc_features].corr()

    fig, ax = plt.subplots(figsize=(max(8, len(mc_features)), max(6, len(mc_features) - 2)))
    sns.heatmap(
        corr_mc,
        annot=True, fmt='.2f',
        cmap='coolwarm', center=0,
        vmin=-1, vmax=1,
        linewidths=0.4,
        ax=ax,
    )
    ax.set_title(f'Multicolinearidade — features com |corr| >= {THRESHOLD} entre si', pad=12)
    plt.tight_layout()
    plt.show()

## Seção 5 — Achados e Hipóteses

Tópicos a cobrir:

- Síntese dos principais padrões observados nas seções anteriores
- Hipóteses sobre features relevantes para o modelo
- Desafios identificados para pré-processamento (Sprint 2): missings críticos, outliers, encoding necessário
- Questões em aberto e decisões a tomar nas próximas sprints

In [ ]:
## codigo